## Analysis of the obtained results

In [43]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()

Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


In [44]:
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )

## Role Playing Core

In [59]:
import pandas as pd
import glob
import os

# Load baseline
file_path = "results/openai_4.1_mini/few_shot/classic/results_stereotype_few_shot_prompt_short_3examples_binary"
folder_path = "results/openai_4.1_mini/few_shot/role_playing_core/*/results_stereotype_few_shot_prompt_short_3examples_binary.csv"

df_base = pd.read_csv(file_path)
merged = df_base[["sample_id", "true_label", "pred_label"]].rename(columns={"pred_label": "base_pred"})

# Load all other result files
profile_files = glob.glob(folder_path)  # adjust path

print(f"🔍 Found {len(profile_files)} profile result files.")

for file in profile_files:
    profile_folder = os.path.basename(os.path.dirname(file))  # e.g., "profile7_passive"
    df_profile = pd.read_csv(file)

    if "sample_id" not in df_profile.columns or "pred_label" not in df_profile.columns:
        print(f"⚠️ Skipping {profile_folder} (missing required columns)")
        continue

    df_profile = df_profile[["sample_id", "pred_label"]].rename(columns={"pred_label": profile_folder})

    # Avoid merge conflicts
    if profile_folder in merged.columns:
        merged = merged.drop(columns=[profile_folder])

    merged = merged.merge(df_profile, on="sample_id", how="left")


sample_mgsd = sample_mgsd.reset_index(drop=True)
sample_mgsd["sample_id"] = sample_mgsd.index

merged = merged.merge(
    sample_mgsd[["sample_id", "stereotype_type", "original_dataset"]],
    on="sample_id",
    how="left"
)

merged["true_label"] = merged["true_label"].astype(str).str.strip().str.lower()
merged["base_pred"] = merged["base_pred"].astype(str).str.strip().str.lower()
for col in merged.columns[3:]:
    merged[col] = merged[col].astype(str).str.strip().str.lower()

fixed_columns = ["sample_id", "true_label", "base_pred"]
profile_columns = sorted([col for col in merged.columns if col not in fixed_columns])
merged = merged[fixed_columns + profile_columns]

print("✅ Merged DataFrame ready with columns:\n", merged.columns.tolist())

🔍 Found 12 profile result files.
✅ Merged DataFrame ready with columns:
 ['sample_id', 'true_label', 'base_pred', 'original_dataset', 'profile1_active', 'profile1_passive', 'profile2_active', 'profile2_passive', 'profile3_active', 'profile3_passive', 'profile4_active', 'profile4_passive', 'profile5_active', 'profile5_passive', 'profile6_active', 'profile6_passive', 'stereotype_type']


In [60]:
from sklearn.metrics import accuracy_score

# Precompute once
base_acc = accuracy_score(merged["true_label"], merged["base_pred"])
print(f"Baseline Accuracy: {base_acc:.2%}")

results = []
profile_cols = [col for col in merged.columns if col.startswith("profile")]

for profile in profile_cols:
    try:
        acc = accuracy_score(merged["true_label"], merged[profile])
        delta = acc - base_acc
        results.append({
            "profile": profile,
            "accuracy": acc,
            "delta_vs_base": delta
        })
    except Exception as e:
        print(f"⚠️ Error computing accuracy for {profile}: {e}")

df_acc = pd.DataFrame(results)
if df_acc.empty:
    print("❌ No valid results to show.")
else:
    df_acc = df_acc.sort_values("delta_vs_base", ascending=False)
    print(df_acc)


Baseline Accuracy: 71.40%
             profile  accuracy  delta_vs_base
5   profile3_passive     0.716          0.002
7   profile4_passive     0.714          0.000
11  profile6_passive     0.712         -0.002
9   profile5_passive     0.710         -0.004
4    profile3_active     0.708         -0.006
1   profile1_passive     0.706         -0.008
6    profile4_active     0.706         -0.008
3   profile2_passive     0.700         -0.014
8    profile5_active     0.694         -0.020
0    profile1_active     0.692         -0.022
2    profile2_active     0.690         -0.024
10   profile6_active     0.684         -0.030


In [61]:
from statsmodels.stats.contingency_tables import mcnemar

print("\n🔍 McNemar Test Results (Profile vs Baseline)\n")

for profile in profile_cols:
    try:
        both_correct = ((merged["base_pred"] == merged["true_label"]) & (merged[profile] == merged["true_label"])).sum()
        base_only    = ((merged["base_pred"] == merged["true_label"]) & (merged[profile] != merged["true_label"])).sum()
        profile_only = ((merged["base_pred"] != merged["true_label"]) & (merged[profile] == merged["true_label"])).sum()
        both_wrong   = ((merged["base_pred"] != merged["true_label"]) & (merged[profile] != merged["true_label"])).sum()

        table = [[both_correct, base_only],
                 [profile_only, both_wrong]]

        result = mcnemar(table, exact=False)
        print(f"{profile}: p={result.pvalue:.4f} (stat={result.statistic:.2f})")
    except Exception as e:
        print(f"⚠️ McNemar error for {profile}: {e}")



🔍 McNemar Test Results (Profile vs Baseline)

profile1_active: p=0.1093 (stat=2.56)
profile1_passive: p=0.6434 (stat=0.21)
profile2_active: p=0.0896 (stat=2.88)
profile2_passive: p=0.3367 (stat=0.92)
profile3_active: p=0.7423 (stat=0.11)
profile3_passive: p=1.0000 (stat=0.00)
profile4_active: p=0.6353 (stat=0.23)
profile4_passive: p=0.8597 (stat=0.03)
profile5_active: p=0.1547 (stat=2.02)
profile5_passive: p=0.8597 (stat=0.03)
profile6_active: p=0.0148 (stat=5.94)
profile6_passive: p=1.0000 (stat=0.00)


In [62]:
print("\n📊 Accuracy by Stereotype Type\n")

for stereotype in merged["stereotype_type"].dropna().unique():
    subset = merged[merged["stereotype_type"] == stereotype]
    print(f"\n=== {stereotype} ===")
    for profile in profile_cols:
        try:
            acc = accuracy_score(subset["true_label"], subset[profile])
            base_acc = accuracy_score(subset["true_label"], subset["base_pred"])
            delta = acc - base_acc
            print(f"{profile}: Δacc = {delta:.2%}")
        except Exception as e:
            print(f"⚠️ Error for {profile} on {stereotype}: {e}")



📊 Accuracy by Stereotype Type


=== race ===
profile1_active: Δacc = -2.05%
profile1_passive: Δacc = -0.41%
profile2_active: Δacc = -1.64%
profile2_passive: Δacc = -1.64%
profile3_active: Δacc = 0.82%
profile3_passive: Δacc = 1.23%
profile4_active: Δacc = 1.64%
profile4_passive: Δacc = 0.41%
profile5_active: Δacc = 0.00%
profile5_passive: Δacc = -0.41%
profile6_active: Δacc = -2.05%
profile6_passive: Δacc = 0.41%

=== profession ===
profile1_active: Δacc = -3.93%
profile1_passive: Δacc = -0.56%
profile2_active: Δacc = -3.93%
profile2_passive: Δacc = -0.56%
profile3_active: Δacc = -2.25%
profile3_passive: Δacc = 0.00%
profile4_active: Δacc = -3.93%
profile4_passive: Δacc = 0.56%
profile5_active: Δacc = -3.93%
profile5_passive: Δacc = 0.56%
profile6_active: Δacc = -4.49%
profile6_passive: Δacc = -0.56%

=== gender ===
profile1_active: Δacc = 1.61%
profile1_passive: Δacc = -1.61%
profile2_active: Δacc = -1.61%
profile2_passive: Δacc = -1.61%
profile3_active: Δacc = -1.61%
profile3_passiv

In [63]:
base_errors = merged[merged["base_pred"] != merged["true_label"]]

rescue_stats = []

for profile in profile_cols:
    # Among base errors, how many this profile gets right?
    rescued = (base_errors[profile] == base_errors["true_label"]).sum()
    total_base_errors = len(base_errors)
    rescue_rate = rescued / total_base_errors

    rescue_stats.append({
        "profile": profile,
        "rescued": rescued,
        "rescue_rate": rescue_rate,
    })

df_rescue = pd.DataFrame(rescue_stats).sort_values("rescue_rate", ascending=False)
#print(df_rescue)

extra_errors = []

for profile in profile_cols:
    profile_errors = (merged[profile] != merged["true_label"]).sum()
    base_errors = (merged["base_pred"] != merged["true_label"]).sum()
    diff = profile_errors - base_errors
    extra_errors.append({
        "profile": profile,
        "extra_errors_vs_base": diff
    })

df_extra = pd.DataFrame(extra_errors)
df_combined = pd.merge(df_rescue, df_extra, on="profile")
print(df_combined.sort_values("rescued", ascending=False))

             profile  rescued  rescue_rate  extra_errors_vs_base
0   profile1_passive       19     0.132867                     4
1    profile4_active       18     0.125874                     4
2    profile3_active       17     0.118881                     3
3   profile2_passive       16     0.111888                     7
4   profile4_passive       16     0.111888                     0
5    profile2_active       15     0.104895                    12
6   profile3_passive       15     0.104895                    -1
7    profile5_active       15     0.104895                    10
8   profile5_passive       15     0.104895                     2
9    profile1_active       14     0.097902                    11
10  profile6_passive       13     0.090909                     1
11   profile6_active        9     0.062937                    15


In [64]:
from scipy.stats import binomtest
import pandas as pd

# Step 1: Compute rescue counts and extra errors
rescued_counts = []
extra_errors_counts = []

for profile in profile_cols:
    rescued = ((merged["base_pred"] != merged["true_label"]) & (merged[profile] == merged["true_label"])).sum()
    extra_errors = ((merged["base_pred"] == merged["true_label"]) & (merged[profile] != merged["true_label"])).sum()
    rescued_counts.append(rescued)
    extra_errors_counts.append(extra_errors)

# Step 2: Build DataFrame
rescue_ratio_df = pd.DataFrame({
    "profile": profile_cols,
    "rescued": rescued_counts,
    "extra_errors": extra_errors_counts
})

# Step 3: Compute rescue rate (rescued / total baseline errors)
total_errors = (merged["base_pred"] != merged["true_label"]).sum()
rescue_ratio_df["rescue_rate"] = rescue_ratio_df["rescued"] / total_errors

# Step 4: Compute ratio of rescued to extra errors (add epsilon to avoid zero division)
rescue_ratio_df["rescue_vs_error_ratio"] = rescue_ratio_df["rescued"] / (rescue_ratio_df["extra_errors"] + 1e-5)

# Step 5: Sort by rescue_vs_error_ratio
rescue_ratio_df = rescue_ratio_df.sort_values("rescue_vs_error_ratio", ascending=False)

# Optional: Add binomial confidence intervals (95%) on rescue_rate
def compute_ci(successes, trials, alpha=0.05):
    result = binomtest(successes, n=trials)
    ci_low, ci_high = result.proportion_ci(confidence_level=1 - alpha)
    return ci_low, ci_high

ci_bounds = [compute_ci(r, total_errors) for r in rescue_ratio_df["rescued"]]
rescue_ratio_df["ci_low"] = [lo for lo, hi in ci_bounds]
rescue_ratio_df["ci_high"] = [hi for lo, hi in ci_bounds]

# Display final results
print(rescue_ratio_df[["profile", "rescued", "rescue_rate", "ci_low", "ci_high", "extra_errors", "rescue_vs_error_ratio"]])


             profile  rescued  rescue_rate    ci_low   ci_high  extra_errors  \
5   profile3_passive       15     0.104895  0.059909  0.167102            14   
7   profile4_passive       16     0.111888  0.065323  0.175334            16   
11  profile6_passive       13     0.090909  0.049297  0.150440            14   
9   profile5_passive       15     0.104895  0.059909  0.167102            17   
4    profile3_active       17     0.118881  0.070800  0.183507            20   
1   profile1_passive       19     0.132867  0.081928  0.199690            23   
6    profile4_active       18     0.125874  0.076337  0.191624            22   
3   profile2_passive       16     0.111888  0.065323  0.175334            23   
8    profile5_active       15     0.104895  0.059909  0.167102            25   
0    profile1_active       14     0.097902  0.054565  0.158806            25   
2    profile2_active       15     0.104895  0.059909  0.167102            27   
10   profile6_active        9     0.0629